# P3 — Los caminos son mensajes

El puente entre P1 (over-squashing) y P2 (conteo de caminos). Tras `g` capas, un mensaje de `i` llega a `j`
**una vez por cada camino de longitud `g`**. Así que `n_g(i,j)` es exactamente *cuántas copias de la
información de `i` llegan a `j`* — todas apretadas en el único vector de `j`.

**Figuras:** (1) acumulación de mensajes vs profundidad, (2) conteo crudo vs de-duplicado, (3) mapas de calor
`A^g` vs `M_g`.

In [ ]:
import torch, numpy as np
from oversquash import viz
from oversquash.data_bottleneck import make_bottleneck_graph
from oversquash.ideal_bridge import walk_operators
K, M = 5, 4

## Figura 1 — los mensajes se acumulan (= el conteo de caminos)

La misma explosión de P1, ahora leída como *multiplicidad de mensajes*: `K·M^d` copias llegan al objetivo.

In [ ]:
depths = [1,2,3,4]; msgs = []
for d in depths:
    dd = make_bottleneck_graph(K, M, d, torch.Generator().manual_seed(0))
    raw, _ = walk_operators(dd.edge_index.numpy(), dd.num_nodes, max_length=d+1)
    t = int(dd.root_mask.nonzero()[0]); msgs.append(int(raw[d][:, t].sum()))
viz.plot_message_explosion(depths, msgs, 'mensajes que llegan al objetivo', 'profundidad', 'mensajes (log)')

## Figura 2 — redundante vs distinto (crudo vs kQ/I)

Muchos de esos mensajes son **redundantes** (nodos intermedios intercambiables llevan el mismo contenido). El
cociente `kQ/I` fusiona caminos equivalentes, colapsando el conteo de `K·M^d` a ~`K`.

In [ ]:
raw_d, eff_d = [], []
for d in depths:
    dd = make_bottleneck_graph(K, M, d, torch.Generator().manual_seed(0))
    raw, eff = walk_operators(dd.edge_index.numpy(), dd.num_nodes, max_length=d+1)
    t = int(dd.root_mask.nonzero()[0])
    raw_d.append(raw[d][:, t].sum()); eff_d.append(eff[d][:, t].sum())
viz.plot_raw_vs_eff(depths, raw_d, eff_d, 'mensajes crudos vs de-duplicados', 'profundidad', 'mensajes (log)',
                    legend=('crudo  (A^g)', 'efectivo  (kQ/I)'))

## Figura 3 — los dos operadores lado a lado

`A^g` (conteo crudo) vs `M_g = dim(e_i·(kQ/I)_g·e_j)` (de-duplicado). Mismo soporte, muchas menos 'copias' a la
derecha.

**El giro (demostrado en P4–P5):** fusionar mensajes redundantes es el arreglo *equivocado* — descarta señal.
El arreglo correcto es conservar todos los caminos pero **aprender cuánto pesar cada uno** (atención).

In [ ]:
dd = make_bottleneck_graph(K, M, 3, torch.Generator().manual_seed(0))
raw, eff = walk_operators(dd.edge_index.numpy(), dd.num_nodes, max_length=4)
viz.plot_two_heatmaps(raw[3], eff[3], titles=('A^4 (crudo)', 'M_4 (kQ/I, de-duplicado)'))